In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
data = pd.read_csv('../datasets/employee.csv')
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


- Convert JoiningYear to datetime, create a 'JoiningDate' column (assume Jan 1)
- Convert PaymentTier to category datatype
- Check and handle any missing values

In [25]:
data['joining_date'] = pd.to_datetime(data['JoiningYear'].astype(str) + '-01-01') 
data['payment_tier'] = data['PaymentTier'].map({1: 'gold', 2: 'silver', 3: 'bronze'}).astype('category')

In [26]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Education                  4653 non-null   object        
 1   JoiningYear                4653 non-null   int64         
 2   City                       4653 non-null   object        
 3   PaymentTier                4653 non-null   int64         
 4   Age                        4653 non-null   int64         
 5   Gender                     4653 non-null   object        
 6   EverBenched                4653 non-null   object        
 7   ExperienceInCurrentDomain  4653 non-null   int64         
 8   LeaveOrNot                 4653 non-null   int64         
 9   joining_date               4653 non-null   datetime64[ns]
 10  payment_tier               4653 non-null   category      
dtypes: category(1), datetime64[ns](1), int64(5), object(4)
memory usage: 

In [30]:
data.isna().sum()

Education                    0
JoiningYear                  0
City                         0
PaymentTier                  0
Age                          0
Gender                       0
EverBenched                  0
ExperienceInCurrentDomain    0
LeaveOrNot                   0
joining_date                 0
payment_tier                 0
dtype: int64

- Create a new column 'TotalExperience' by calculating years from JoiningYear to current year
- Create 'ExperienceLevel' column: 'Junior' (<3 years), 'Mid' (3-7 years), 'Senior' (>7 years)

In [38]:
current_year = pd.Timestamp.now().year
data['total_experience'] = current_year - data['JoiningYear']
data['experience_level'] = np.where(data['total_experience']<5, 'junior', np.where(data['total_experience']<=10, 'mid', 'senior'))

In [39]:
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0,2017-01-01,bronze,9,mid
1,Bachelors,2013,Pune,1,28,Female,No,3,1,2013-01-01,gold,13,senior
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0,2014-01-01,bronze,12,senior
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid
4,Masters,2017,Pune,3,24,Male,Yes,2,1,2017-01-01,bronze,9,mid


- Calculate the average LeaveOrNot rate by:

    (Education level, City, Gender)

- Find which combination has the highest attrition rate

In [51]:
leave_summary = data.groupby(['Education', 'City', 'Gender']).agg(
    leave_rate = ('LeaveOrNot', 'mean'),
    total_employees = ('LeaveOrNot', 'count'),
    left_count = ('LeaveOrNot', 'sum')
)
leave_summary['leave_rate%'] = leave_summary['leave_rate'].apply(lambda x: f"{x:0.2%}")
leave_summary

leave_rate  total_employees  left_count  \
Education City      Gender                                            
Bachelors Bangalore Female    0.265651              591         157   
                    Male      0.229979             1461         336   
          New Delhi Female    0.234637              358          84   
                    Male      0.072626              179          13   
          Pune      Female    0.940329              486         457   
                    Male      0.155894              526          82   
Masters   Bangalore Female    0.703704               54          38   
                    Male      0.700000               70          49   
          New Delhi Female    0.458333              216          99   
                    Male      0.478405              301         144   
          Pune      Female    0.306931              101          31   
                    Male      0.496183              131          65   
PHD       Bangalore Female    0.214286               14           3   
                    Male      0.315789               38          12   
          New Delhi Female    0.260870               46          12   
                    Male      0.245614               57          14   
          Pune      Female    0.333333                9           3   
                    Male      0.066667               15           1   

                           leave_rate%  
Education City      Gender              
Bachelors Bangalore Female      26.57%  
                    Male        23.00%  
          New Delhi Female      23.46%  
                    Male         7.26%  
          Pune      Female      94.03%  
                    Male        15.59%  
Masters   Bangalore Female      70.37%  
                    Male        70.00%  
          New Delhi Female      45.83%  
                    Male        47.84%  
          Pune      Female      30.69%  
                    Male        49.62%  
PHD       Bangalore Female      21.43%  
                    Male        31.58%  
          New Delhi Female      26.09%  
                    Male        24.56%  
          Pune      Female      33.33%  
                    Male         6.67%

In [56]:
leave_summary['leave_rate'].idxmax()

('Bachelors', 'Pune', 'Female')

In [55]:
leave_summary.loc[leave_summary['leave_rate'].idxmax()]

leave_rate         0.940329
total_employees         486
left_count              457
leave_rate%          94.03%
Name: (Bachelors, Pune, Female), dtype: object

- Find all employees who:
- - Have Masters degree
- - Are from Bangalore or Pune
- - Have >3 years experience
- - Are NOT benched

In [57]:
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0,2017-01-01,bronze,9,mid
1,Bachelors,2013,Pune,1,28,Female,No,3,1,2013-01-01,gold,13,senior
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0,2014-01-01,bronze,12,senior
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid
4,Masters,2017,Pune,3,24,Male,Yes,2,1,2017-01-01,bronze,9,mid


In [60]:
mask_city = (data['City']=='Bangalore') | (data['City']=='Pune')
mask = (data['Education']=='Masters') & (mask_city) & (data['total_experience']>3) & (data['EverBenched']=='No')
data.loc[mask].head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid
10,Masters,2012,Bangalore,3,27,Male,No,5,1,2012-01-01,bronze,14,senior
57,Masters,2014,Pune,3,39,Female,No,2,0,2014-01-01,bronze,12,senior
59,Masters,2017,Pune,2,36,Male,No,2,1,2017-01-01,silver,9,mid
69,Masters,2017,Bangalore,3,40,Female,No,2,1,2017-01-01,bronze,9,mid


- Create a 'RiskScore' column using a function:
- - Start with 0
- - +1 if Age > 35
- - +1 if EverBenched is Yes
- - +1 if ExperienceInCurrentDomain < 3
- - +1 if PaymentTier is 1 or 2
- Categorize as 'Low' (0-1), 'Medium' (2), 'High' (3-4)